# 8.1 Callback

使用 Keras callbacks 監控訓練、提早停止、儲存最佳模型、調整 learning rate，並保存訓練紀錄。

## 1. 任務說明

Callback 是 Keras 訓練流程中的自動化控制工具。它不改變模型架構，而是在訓練過程中根據 loss、accuracy 或 validation 指標執行指定動作。

本 notebook 使用 Fashion-MNIST 小型子集訓練 CNN，示範四個常用 callback：

- `EarlyStopping`：validation loss 長時間沒有改善時停止訓練。
- `ModelCheckpoint`：只保存 validation loss 最好的模型。
- `ReduceLROnPlateau`：模型停滯時降低 learning rate。
- `CSVLogger`：把每個 epoch 的訓練紀錄寫入 CSV。

## 2. 載入套件

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow version:", tf.__version__)

## 3. 載入資料

Fashion-MNIST 是 10 類灰階服飾圖片資料集。為了讓範例在 Colab CPU 或一般筆電上快速完成，本篇只取部分資料進行示範。

In [ ]:
(x_train_full, y_train_full), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

train_size = 12000
test_size = 3000

x_train = x_train_full[:train_size].astype("float32") / 255.0
y_train = y_train_full[:train_size]
x_test = x_test[:test_size].astype("float32") / 255.0
y_test = y_test[:test_size]

x_train = x_train[..., np.newaxis]
x_test = x_test[..., np.newaxis]

print("x_train:", x_train.shape)
print("x_test:", x_test.shape)

## 4. 觀察資料

In [ ]:
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i].squeeze(), cmap="gray")
    plt.title(class_names[y_train[i]])
    plt.axis("off")
plt.tight_layout()
plt.show()

## 5. 建立模型

為了讓 callback 的效果容易觀察，這裡建立一個小型 CNN。函式化建立模型可以讓後續重複實驗更方便。

In [ ]:
def build_model():
    tf.keras.utils.set_random_seed(42)

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(28, 28, 1)),
        tf.keras.layers.Conv2D(32, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(10, activation="softmax"),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

model = build_model()
model.summary()

## 6. 建立 callbacks

`monitor="val_loss"` 表示根據 validation loss 判斷是否改善。若訓練資料沒有切出 validation set，就不能使用 `val_loss` 作為監控指標。

In [ ]:
artifact_dir = Path("artifacts/8_1_callbacks")
artifact_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = artifact_dir / "best_model.keras"
log_path = artifact_dir / "training_log.csv"

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_path,
        monitor="val_loss",
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-5,
        verbose=1,
    ),
    tf.keras.callbacks.CSVLogger(log_path),
]

callbacks

## 7. 訓練模型

`callbacks=callbacks` 會把前一節建立的 callback 串進訓練流程。這裡刻意設定較高的 `epochs`，讓 `EarlyStopping` 有機會在模型停滯時提前結束。

In [ ]:
history = model.fit(
    x_train,
    y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=128,
    callbacks=callbacks,
    verbose=1,
)

history_df = pd.DataFrame(history.history)
history_df.tail()

## 8. 視覺化訓練紀錄

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

history_df[["loss", "val_loss"]].plot(ax=axes[0], title="Loss")
history_df[["accuracy", "val_accuracy"]].plot(ax=axes[1], title="Accuracy")

for ax in axes:
    ax.set_xlabel("epoch")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. 檢查 callback 產物

`ModelCheckpoint` 會輸出最佳模型，`CSVLogger` 會輸出訓練紀錄。這些檔案可以用於後續比較、恢復模型或保存實驗紀錄。

In [ ]:
print("checkpoint exists:", checkpoint_path.exists(), checkpoint_path)
print("log exists:", log_path.exists(), log_path)

log_df = pd.read_csv(log_path)
log_df.tail()

## 10. 評估模型

因為 `EarlyStopping(restore_best_weights=True)` 會把模型權重恢復到 validation loss 最好的 epoch，目前記憶體中的 `model` 已經是較佳權重。也可以另外從 `best_model.keras` 載入 checkpoint 進行評估。

In [ ]:
train_loss, train_acc = model.evaluate(x_train, y_train, verbose=0)
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)

pd.DataFrame({
    "dataset": ["train", "test"],
    "loss": [train_loss, test_loss],
    "accuracy": [train_acc, test_acc],
})

In [ ]:
best_model = tf.keras.models.load_model(checkpoint_path)
best_test_loss, best_test_acc = best_model.evaluate(x_test, y_test, verbose=0)
print("best checkpoint test accuracy:", round(best_test_acc, 4))

## 11. 混淆矩陣與分類報告

In [ ]:
y_prob = best_model.predict(x_test, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=class_names))

## 12. 如何套用自己的資料？

替換成自己的任務時，callback 大多可以保留，主要修改以下位置：

1. `monitor`：常用 `val_loss` 或 `val_accuracy`，但該指標必須存在於訓練紀錄中。
2. `checkpoint_path`：改成專案中要保存模型的位置。
3. `patience`：資料小或 validation 指標波動大時可調大。
4. `min_lr`：若使用 `ReduceLROnPlateau`，需設定合理的 learning rate 下限。

## 13. 小結

Callback 讓訓練流程更接近實務工作：訓練過程可以自動停止、保存、調整 learning rate 與留下紀錄。當模型與資料變大時，callback 通常不是選配，而是標準訓練流程的一部分。